In [1]:
import os
import sys

# Third-party numerical and data handling
import numpy as np
import pandas as pd
# Visualization
import matplotlib.pyplot as plt

# PyTorch core
import torch
from torch import nn
from torch.utils.data import DataLoader

from model_evaluation_helpers import check_device, read_preprocessed_images, ECGDataset, val_transforms, MultiHeadEfficientNet, get_probs_and_labels, compute_ranking_metrics
from sklearn.model_selection import train_test_split

In [2]:
# Check and get device
device = check_device()

try: 
    print(image_set["train_000000.png"])
except Exception as e:
    image_set = read_preprocessed_images("/Users/griffinfarrow/Documents/Data_Science_Projects/bhf_classification/data/proc_images.h5")
    

label_df = pd.read_csv("/Users/griffinfarrow/Documents/Data_Science_Projects/bhf_classification/data/train_final.csv", index_col=0)

image_names = list(image_set.keys())
image_ids = list([int(image_name.split(".")[0][-6:]) for image_name in image_names])
label_df = label_df.loc[label_df.index.isin(set(image_ids))]

# need to order label_df so that it has the same ordering as image_names
image_df = pd.DataFrame({
    "image_name": image_set.keys(),
})
image_df["image_id"] = image_df["image_name"].str.split(".", expand=True)[0].str[-6:].astype(int)
image_df = image_df.sort_values(by="image_id")
image_df = image_df.set_index("image_id")
# ensure ordering of label_df and image_df
label_df = label_df.loc[image_df.index]
X_train, X_test, y_train, y_test = train_test_split(image_df,
                                                    label_df,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    shuffle=True,
                                                    stratify=label_df[["CD", "MI", "AF", "STTC", "HYP"]]
                                                    )

train_image_names = X_train["image_name"].tolist()
test_image_names = X_test["image_name"].tolist()

val_dataset = ECGDataset(
    image_names = test_image_names,
    image_set = image_set,
    labels_df = y_test,
    transforms = val_transforms
)

num_workers = 0 if sys.platform == 'darwin' else 4 
print(f"Using num_workers = {num_workers}")

val_dataloader = DataLoader( 
                        val_dataset,
                        batch_size=32, 
                        shuffle=False, 
                        num_workers=num_workers, 
                        pin_memory=True if device.type == "cuda" else False)

✓ MPS (Apple Silicon GPU) available

Selected device: mps
Using num_workers = 0


In [3]:
def evaluate_model(modelpath, modeltype, verbose=True):
    checkpoint = torch.load(modelpath, map_location=torch.device("cpu"), weights_only=False)
    current_epoch = checkpoint["epoch"]
    if verbose:
        print(f"Loading from checkpoint, last run epoch was {current_epoch}")
        
    model = MultiHeadEfficientNet(
        num_conditions=5, 
        hidden_dim=512, 
        dropout_rate=0.3, 
        model=modeltype
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"])
    val_probs, val_labels = get_probs_and_labels(loader=val_dataloader, model=model, device=device)
    ranking_metrics = compute_ranking_metrics(val_labels, val_probs)
    
    return ranking_metrics

In [4]:
def evaluate_multiple_models(modeldict, modeltype="convnext"):
    """
    Takes in dictionary with format {str: str}, representing {modelname: path to model checkpoint} and collects ranking metrics
    for those models
    """
    collected_results = {}
    for modelname, modelpath in modeldict.items(): 
        results = evaluate_model(modelpath, modeltype)
        collected_results[modelname] = results
    return collected_results

In [5]:
basepath = os.path.join(os.path.expanduser("~"), "Library", "CloudStorage", "OneDrive-Nexus365", "BHF_Cardiac_Problem", "Results_Collection")

modeldict = {
    "Asymmetric": os.path.join(basepath, "Model_Asymm_Loss", "best_model.pth"),
    "Focal": os.path.join(basepath, "Model_Focal_Loss", "best_model.pth"), 
    "BCE_Weighted": os.path.join(basepath, "Model_BCE_Weighted", "best_model.pth"), 
    "BCE_Unweighted": os.path.join(basepath, "Model_BCE_Unweighted", "best_model.pth")
}

agg_results = evaluate_multiple_models(modeldict)

Loading from checkpoint, last run epoch was 10


Loading from checkpoint, last run epoch was 12


Loading from checkpoint, last run epoch was 14


Loading from checkpoint, last run epoch was 12


In [6]:
from tabulate import tabulate
models = agg_results.keys()
to_write = []
for model in models: 
    to_write.append([model, 
                    agg_results[model]["macro_f1"],
                    agg_results[model]["macro_precision"], 
                    agg_results[model]["macro_recall"], 
                    agg_results[model]["macro_ece"], 
                    agg_results[model]["micro_ece"],
                    agg_results[model]["macro_brier"], 
                    agg_results[model]["macro_auroc"], 
                    agg_results[model]["micro_auroc"], 
                    agg_results[model]["macro_ap"], 
                    agg_results[model]["micro_ap"]])
    
print(tabulate(to_write, headers=["Model Name", "Macro F1", "Macro_Precision", "Macro_Recall", "Macro_ECE", "Micro_ECE", "Macro_Brier", "Macro_AUROC", "Micro_AUROC", "Macro_AP", "Micro_AP"]))

Model Name        Macro F1    Macro_Precision    Macro_Recall    Macro_ECE    Micro_ECE    Macro_Brier    Macro_AUROC    Micro_AUROC    Macro_AP    Micro_AP
--------------  ----------  -----------------  --------------  -----------  -----------  -------------  -------------  -------------  ----------  ----------
Asymmetric        0.749971           0.746866        0.755761    0.246063     0.24361        0.136713        0.934602       0.939789    0.817135    0.821385
Focal             0.754467           0.733244        0.778657    0.0963174    0.0881873      0.0784002       0.937303       0.941022    0.820267    0.813303
BCE_Weighted      0.764222           0.752197        0.780222    0.127874     0.127874       0.0915875       0.938777       0.934551    0.820937    0.798867
BCE_Unweighted    0.760418           0.764217        0.759018    0.0262163    0.0137868      0.0663347       0.937147       0.941755    0.820609    0.822946


In [7]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "f1"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name        STTC_f1    HYP_f1     MI_f1     CD_f1     AF_f1
--------------  ---------  --------  --------  --------  --------
Asymmetric       0.750983  0.626667  0.754278  0.77      0.847926
Focal            0.748538  0.618529  0.756656  0.765854  0.882759
BCE_Weighted     0.758318  0.653686  0.747849  0.777961  0.883295
BCE_Unweighted   0.758621  0.658635  0.758621  0.772496  0.853717


In [8]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "precision"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name        STTC_precision    HYP_precision    MI_precision    CD_precision    AF_precision
--------------  ----------------  ---------------  --------------  --------------  --------------
Asymmetric              0.717146         0.618421        0.788269        0.803478        0.807018
Focal                   0.70936          0.623626        0.716295        0.778512        0.838428
BCE_Weighted            0.69746          0.673352        0.754339        0.800338        0.835498
BCE_Unweighted          0.719753         0.65252         0.782427        0.822785        0.843602


In [9]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "recall"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name        STTC_recall    HYP_recall    MI_recall    CD_recall    AF_recall
--------------  -------------  ------------  -----------  -----------  -----------
Asymmetric           0.788171      0.635135     0.723097       0.7392     0.893204
Focal                0.792297      0.613514     0.801837       0.7536     0.932039
BCE_Weighted         0.830812      0.635135     0.74147        0.7568     0.936893
BCE_Unweighted       0.801926      0.664865     0.73622        0.728      0.864078


In [10]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "auroc"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name        STTC_auroc    HYP_auroc    MI_auroc    CD_auroc    AF_auroc
--------------  ------------  -----------  ----------  ----------  ----------
Asymmetric          0.934135     0.903932    0.920489    0.92802     0.986435
Focal               0.933419     0.911172    0.919838    0.932003    0.990085
BCE_Weighted        0.935919     0.916597    0.920833    0.934481    0.986055
BCE_Unweighted      0.937518     0.912181    0.919917    0.927945    0.988173
